When we write the data , it'll create the Parquet files. 
Let's say our pipelines runs frequently, it'll create so many files 
![image_1773505183845.png](./image_1773505183845.png "image_1773505183845.png")

Problem : Engine says it'll always to better to read larger but lesser files as compare to smaller and many files. 
We can't create a single file. Engines are built for big data 

CREATE A DIRECTORY IN THE BRONZ VOLUME

delta optimization

![image_1773505467744.png](./image_1773505467744.png "image_1773505467744.png")

In [0]:
data = [(1,100), (2,200), (3,300)]
df = spark.createDataFrame(data, ["id", "salary"])

In [0]:
display(df)

Re run to create some files . Let's say we run this 5 times , 5 files will be created

In [0]:
df.write.format("delta").mode("append").save("/Volumes/databricksharis/bronze/bronze_volume/deltaopt/")

Let's re run the files for 5times, We should have like 5 files 

In [0]:
%sql
select * from delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt`

### **OPTIMZE** : IT'LL COALESCE SMALL FILES INTO BIG FILE

In [0]:
display(dbutils.fs.ls("/Volumes/databricksharis/bronze/bronze_volume/deltaopt/"))

In [0]:
%sql
OPTIMIZE delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

It Creates a file by coalascing the other files,  which is havimg data of other files
this information is saved in Delta Log

In [0]:
display(dbutils.fs.ls("/Volumes/databricksharis/bronze/bronze_volume/deltaopt/"))

In [0]:
%sql

----it takes 4 secs instead of 6 secs
select * from delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`


It's cherry on Cake, It sorts the data within the Partition
## ZORDER : BY USING THIS IT'LL ALSO STORE THE COLUMN STATISTICS, by using that we optimize the optimize command

- if we have thousand of files, it'll help in data skipping

![image_1773505907985.png](./image_1773505907985.png "image_1773505907985.png")

As when user raise a query, it doesn't know in which file data is present, so it read it's both files, by zorder, it stores statistics

In [0]:
%sql
OPTIMIZE delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/` ZORDER BY id

In [0]:
%sql
select * from delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

## **LIQUID CLUSTERING**
![image_1773506068445.png](./image_1773506068445.png "image_1773506068445.png")
- IN THIS IT'LL CREATE LOGICAL CLUSTERS BASED OF values/ Columns , IT'LL AUTOMATICALLY EVOLVING, AS PARTITIONS BECOMES OUTDATED AFTER THIS, based on the query , it'll use this and create a cluster on that

In [0]:
%sql
CREATE TABLE databricksharis.bronze.liquid_tbl (
  id INT, 
  salary INT
)

CLUSTER BY (id)

## **TURN OFF THE CLUSTER**

In [0]:
%sql
ALTER TABLE databricksharis.bronze.liquid_tbl
CLUSTER BY NONE

IF YOU NOT SURE WHICH COLUMNS ARE BEST FOR CLUSTER, IT'LL TAKE FROM QUERY BEHAVIOUR AND IS CONTINUOUSLY EVOLVING

In [0]:
%sql
CREATE TABLE databricksharis.bronze.liquid_tbl2 (
  id INT, 
  salary INT
)

CLUSTER BY AUTO

## **DATA VERSIONING**
Everything is recorded whatever we done on table
![image_1773506382461.png](./image_1773506382461.png "image_1773506382461.png")

Delta Lake Gives us Commit and Rollback, delta lake record versions on the top of table

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

## **TIME TRAVEL**
In short if we can go back to time
Insteaf of Delete with Where condition , we run Deleta Table, it'll delete all the records


![image_1773506588997.png](./image_1773506588997.png "image_1773506588997.png")

In [0]:
%sql
DELETE FROM delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

In [0]:
%sql
SELECT * FROM delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

In [0]:
%sql

----DELETE ALSO LOGS HERE 
DESCRIBE HISTORY delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

In [0]:
%sql
RESTORE delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/` TO VERSION AS OF 5

In [0]:
%sql
SELECT * FROM delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`

## **VACUUM** 
- Delta log Keeps the log of 3o days, It'll remove all unessary files which is not being used in Delta Lake
- THIS COMMAND WILL DELETE ALL THE UNUSED FILES, IT'LL REMOVE ALL THE FILES WHICH ARE OLDER THAN 7 DAYS
- DELTA LOG KEEP THE LOG FOR 30 DAYS, BY DEAFULT IT'S 7 DAYS

In [0]:
display(dbutils.fs.ls("/Volumes/databricksharis/bronze/bronze_volume/deltaopt/"))

In [0]:
%sql
VACUUM delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/`
DRY RUN --- USED FOR PREVIEW WHICH FILES TO BE DELETED, NO FILES IS OLDER THAN 7 DAYS THAT' WHY IT'S SHOWING 0 FILES TO BE DELETED
    


In [0]:
display(dbutils.fs.ls("/Volumes/adb_training_haris/bronze/bronze_volume/deltaopt/"))

In [0]:
%sql
VACUUM delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/` RETAIN 0 HOURS --- DELETES ALL FILES OLDER THAN 0 HOURS, we cannot Retain the fils


NEED TO TURN OFF THE CHECK

NOT AVAILABLE TO SERVERLESS COMPUTE, WHICH IS AVAILABLE IN ALL PURPOSE COMPUTE

In [0]:
%sql
-- SET spark.databricks.delta.retentionDurationCheck.enabled = false;

VACUUM delta.`/Volumes/databricksharis/bronze/bronze_volume/deltaopt/` RETAIN 0 HOURS

## **CTAS (CREATE TABLE AS SELECT)**